# Snake Game Using Pygame

This project implements a Snake game using Pygame. The player collects apples, increases the snake's length and score, and avoids obstacles. A second snake becomes active when the player's score exceeds 10.

## 1. Import Libraries and Initialize Pygame

Import the required libraries and initialize the game window, clock, colors, and basic settings.

In [ ]:
import os
import random
import pygame

pygame.init()

try:
    pygame.mixer.init()
    SOUND_ENABLED = True
except pygame.error:
    SOUND_ENABLED = False

SCREEN_WIDTH = 1000
SCREEN_HEIGHT = 1000
BLOCK_SIZE = 100
FPS = 10
ASSET_DIR = "assets"

screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Snake")
clock = pygame.time.Clock()
font = pygame.font.SysFont(None, 25)

red = (220, 20, 60)
white = (255, 255, 255)
background = (200, 200, 200)

## 2. Load Game Assets

Load the images and sound effect from the assets folder. The image loader checks whether each required file exists before loading it.

In [ ]:
def load_image(filename, size=(100, 100)):
    path = os.path.join(ASSET_DIR, filename)
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Asset not found: {path}")
    image = pygame.image.load(path).convert_alpha()
    return pygame.transform.scale(image, size)

image = load_image("circle.png")
image2 = load_image("Golden_Big_Chain_Chomp_Sprite_PM5.png")
image3 = load_image("Featured-Image.png")
image4 = load_image("super_mario_world_mushroom_sprite_by_adventuresmg64gaming-db12tjs.png")
apple_image = load_image("apple90.png")

pygame.display.set_icon(pygame.image.load(
    os.path.join(ASSET_DIR, "Golden_Big_Chain_Chomp_Sprite_PM5.png")
).convert_alpha())

eat_sound = None
sound_path = os.path.join(ASSET_DIR, "burger (1).wav")
if SOUND_ENABLED and os.path.isfile(sound_path):
    eat_sound = pygame.mixer.Sound(sound_path)

def play_eat_sound():
    if eat_sound is not None:
        eat_sound.play()

## 3. Define Helper Functions

These functions generate random positions, create snake segments, display messages and scores, draw the snakes, and place a new apple.

In [ ]:
def random_position():
    x = random.randrange(100, 900, BLOCK_SIZE)
    y = random.randrange(100, 900, BLOCK_SIZE)
    return x, y

def create_segment(x, y):
    rect = image.get_rect()
    rect.topleft = (x, y)
    return rect

def message(msg, color, y_displace=0):
    text_surface = font.render(msg, True, color)
    text_rect = text_surface.get_rect()
    text_rect.center = (SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 + y_displace)
    screen.blit(text_surface, text_rect)

def draw_score(value, x=0):
    text = font.render("Score: " + str(value), True, red)
    screen.blit(text, (x, 0))

def draw_snake(body, head_image, body_image):
    for segment in body[:-1]:
        screen.blit(body_image, segment)
    if body:
        screen.blit(head_image, body[-1])

def spawn_apple(apple):
    apple.topleft = random_position()

## 4. Initialize Game Objects

Create the player snake, the second snake, the apple, the bonus object, and the obstacle. Initialize scores and movement directions.

In [ ]:
player_body = [
    create_segment(700, 400),
    create_segment(600, 400),
    create_segment(500, 400),
    create_segment(400, 400)
]

second_body = [
    create_segment(800, 500),
    create_segment(700, 500),
    create_segment(600, 500),
    create_segment(500, 500)
]

apple = apple_image.get_rect()
spawn_apple(apple)

bonus = image4.get_rect()
bonus.topleft = random_position()

obstacle = image3.get_rect()
obstacle.topleft = (200, 200)

player_score = 0
second_score = 0

# Directions: 0 = down, 1 = right, 2 = up, 3 = left
player_direction = 1
second_direction = 1

game_running = True
game_started = False

## 5. Display the Welcome Screen

Show the welcome screen and wait for the player to press P to start or Q/Escape to quit.

In [ ]:
screen.fill(white)
message("Welcome to Snake", red, -90)
message("Eat the Apple!", red, -40)
message("Press P to play or Q to quit!", red, 10)
pygame.display.update()

while not game_started and game_running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            game_running = False
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_p:
                game_started = True
            elif event.key in (pygame.K_q, pygame.K_ESCAPE):
                game_running = False

## 6. Define Player Movement Functions

The player controls the snake using the arrow keys. The snake cannot immediately reverse its direction. The movement function supports growing the snake by retaining its tail when it eats an apple.

In [ ]:
def handle_player_direction():
    global player_direction
    keys = pygame.key.get_pressed()

    if keys[pygame.K_LEFT] and player_direction in (0, 2):
        player_direction = 3
    elif keys[pygame.K_RIGHT] and player_direction in (0, 2):
        player_direction = 1
    elif keys[pygame.K_DOWN] and player_direction in (1, 3):
        player_direction = 0
    elif keys[pygame.K_UP] and player_direction in (1, 3):
        player_direction = 2

def get_movement(direction):
    movements = {
        0: (0, BLOCK_SIZE),
        1: (BLOCK_SIZE, 0),
        2: (0, -BLOCK_SIZE),
        3: (-BLOCK_SIZE, 0)
    }
    return movements.get(direction, (0, 0))

def move_snake(body, direction, grow=False):
    move_x, move_y = get_movement(direction)
    new_head = create_segment(
        body[-1].x + move_x,
        body[-1].y + move_y
    )
    body.append(new_head)
    if not grow:
        body.pop(0)
    return new_head

## 7. Define the Second Snake Movement

The second snake attempts to move toward the apple. Its direction is updated according to the relative position of the apple and its head, while avoiding an immediate reverse direction.

In [ ]:
def update_second_direction():
    global second_direction

    head = second_body[-1]
    dx = apple.x - head.x
    dy = apple.y - head.y

    if second_direction in (1, 3):
        if dy > 0:
            second_direction = 0
        elif dy < 0:
            second_direction = 2
        elif dx > 0:
            second_direction = 1
        elif dx < 0:
            second_direction = 3
    else:
        if dx > 0:
            second_direction = 1
        elif dx < 0:
            second_direction = 3
        elif dy > 0:
            second_direction = 0
        elif dy < 0:
            second_direction = 2

## 8. Main Game Loop

Update the player and second snake, handle apple and bonus collection, check collisions, draw the game objects, and refresh the display. The snake's growth is now handled directly inside the movement function.

In [ ]:
while game_running and game_started:
    clock.tick(FPS)

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            game_running = False
        elif event.type == pygame.KEYDOWN:
            if event.key in (pygame.K_ESCAPE, pygame.K_q):
                game_running = False

    if not game_running:
        break

    screen.fill(background)
    screen.blit(apple_image, apple)
    screen.blit(image3, obstacle)
    screen.blit(image4, bonus)

    # Update player snake.
    handle_player_direction()
    move_x, move_y = get_movement(player_direction)
    next_player_head = create_segment(
        player_body[-1].x + move_x,
        player_body[-1].y + move_y
    )
    player_eats_apple = next_player_head.colliderect(apple)
    player_head = move_snake(
        player_body, player_direction, grow=player_eats_apple
    )

    if player_eats_apple:
        player_score += 10
        play_eat_sound()
        spawn_apple(apple)

    if player_head.colliderect(bonus):
        player_score -= 10
        bonus.topleft = random_position()

    # Update second snake after the player's score exceeds 10.
    if player_score > 10:
        update_second_direction()
        move_x, move_y = get_movement(second_direction)
        next_second_head = create_segment(
            second_body[-1].x + move_x,
            second_body[-1].y + move_y
        )
        second_eats_apple = next_second_head.colliderect(apple)
        second_head = move_snake(
            second_body, second_direction, grow=second_eats_apple
        )

        if second_eats_apple:
            second_score += 10
            play_eat_sound()
            spawn_apple(apple)

        if second_head.colliderect(bonus):
            bonus.topleft = random_position()

    # Check game-over conditions.
    if player_head.colliderect(obstacle) or player_score <= -50:
        screen.fill(white)
        message("Game Over!", red, -90)
        pygame.display.update()
        pygame.time.wait(1500)
        game_running = False
        break

    # Draw snakes and scores.
    draw_snake(player_body, image2, image)

    if player_score > 10:
        draw_snake(second_body, image2, image)

    draw_score(player_score, 0)
    if player_score > 10:
        draw_score(second_score, 100)

    pygame.display.update()

pygame.quit()

## 9. Execution Notes

Place all required image and sound files in a folder named `assets`, located next to this notebook. Run the notebook cells from top to bottom. Pygame opens a separate graphical window, so running this notebook locally is recommended.